Before we head into example 4, we need to check...
# Readback and writeback: a BRAM with built-in ECC under SK-RAFT injection

This part of [example 4](../README.md) shows two effects of an SK-RAFT injection into the content of a block RAM that uses the built-in ECC of the BRAM and has no output register (`DO_REG = 0`):

1. The configuration access overwrites the output latch of the BRAM. While the clock is stopped, the BRAM outputs a wrong word.
2. When the clock starts, the BRAM reads the word again. The word has the injected bit flipped, and the ECC does not detect it (`SBITERR = 0`).

## Design

`bram_readback_writeback_top` ([`hw/sources_1/new/bram_readback_writeback_top.vhd`](../../hw/sources_1/new/bram_readback_writeback_top.vhd)) has one RAMB36 in simple dual-port mode, 512 x 72 (64 data bits and 8 ECC bits). The built-in ECC encoder makes the ECC bits on each write (`EN_ECC_WRITE = TRUE`). The built-in decoder checks each read (`EN_ECC_READ = TRUE`). The read port is always enabled, so the BRAM reads the word at the address on each clock edge. The BRAM outputs go directly to USER3, with no register in between.

| Register | Width | Access | Contents |
| --- | --- | --- | --- |
| USER1 | 1 bit | write | Clock enable of the BRAM (a BUFGCE). 0 after programming. |
| USER2 | 16 bits | write | Address (only the low 9 bits have an effect). |
| USER3 | 80 bits | write | Bits 63:0 are written to the word at the address. |
| USER3 | 80 bits | read | Bits 63:0: data. Bits 71:64: ECC parity. Bit 72: `SBITERR`. |

The XDC excludes the BRAM from power optimization (`set_power_opt -exclude_cells`). Without it, `opt_design` gates the read enable, and the BRAM does not read again in stage 4.

## Stages

The script writes the address as the word: word 0 holds 0, word 1 holds 1, and so on. The script reads word 0 after each stage.

| Stage | Action | Result |
| --- | --- | --- |
| 1 | Start the clock, fill the BRAM, select word 0. | The word is correct, `SBITERR = 0`. |
| 2 | Stop the clock. | The word is still correct. The output latch holds the last read. |
| 3 | Inject into word 0, bit 0 (`Ram=B:BIT0` in the LL file). The clock is still stopped. | A wrong word. |
| 4 | Start the clock. | Bit 0 is flipped, `SBITERR = 0`. |

**Stage 3.** To flip one bit, SK-RAFT reads the content frame, changes the bit and writes the frame back. This access to a content frame overwrites the output latch of the BRAM port. The BRAM has no output register, so its outputs show the latch directly: data, parity and `SBITERR` are not the values of word 0. The same occurs for each readback and each revert of a content frame.

**Stage 4.** The BRAM reads word 0 again, and the outputs are correct for the stored word. But the frame write went through the ECC encoder of the BRAM, which made new ECC bits for the flipped data. Thus the stored word is a valid ECC word, and the decoder finds no error. The flipped bit is silent data corruption, not an error that the ECC corrects.

Two consequences for SK-RAFT campaigns:

- A design that reads a BRAM with `DO_REG = 0` can take the wrong word of stage 3 after a freeze. The BRAM hold register pass ([`auxiliary_components/bram_hold_pass`](../../../auxiliary_components/bram_hold_pass/README.md)) protects against this. This example builds without the pass to show the effect.
- `TargetMemory.BRAM` injection into a BRAM with `EN_ECC_WRITE = TRUE` does not test the built-in ECC. It gives a valid word with wrong data.

## The calls of the example

| Call | Use in the example |
| --- | --- |
| `SkraftFpga.connect(ip=..., port=...)`, `client.attach(SERIAL_NUMBER)` | Connect to the SK-RAFT server and attach to the device, in `with` statements that release the device and close the connection. |
| `Design(bit, ll)`, `dev.uploadDesignToServer(design)`, `dev.programFPGA(design)` | Give the design files, upload them to the server and program the FPGA. |
| `BscanPrimitive(user, length)`, `dev.write_bscan(primitive, value)`, `dev.read_bscan(primitive)` | Start and stop the clock (USER1), select the address (USER2), write the words and read the BRAM outputs (USER3). |
| `dev.inject_at(far, word, bit, TargetMemory.BRAM)` | Stage 3: inject into word 0, bit 0. The script finds the location of the bit (`Ram=B:BIT0`) in the LL file. |

## Requirements

- Python 3.9 or a later version, with the `skraft` package installed.
- An SK-RAFT server in operation, with the target board connected through USB.
- The bitstream and the LL file of `bram_readback_writeback_top` in `hw/design_examples/<board>/bram_readback_writeback_top/`. Extract them from `design_examples.zip` (refer to [Prebuilt design files](../../README.md#prebuilt-design-files))

On Linux

In [3]:
%%bash
cd ../hw/design_examples/
./unzip_designs.sh

Design files extracted to /home/wallmenis/git/sk-raft-client/get_started/hw/design_examples


On Windows

In [ ]:
%%script pwsh
cd ..\hw\design_examples\
.\unzip_designs.bat

- or build them:

On Linux (don't forget to put the right path for vivado)

In [ ]:
%%bash
cd ../hw
vivado -mode batch -source create_project.tcl -tclargs basys3 bram_readback_writeback_top

On Windows

In [ ]:
%%script pwsh
cd ..\hw
vivado -mode batch -source create_project.tcl -tclargs basys3 bram_readback_writeback_top

## Run the example


Install the toolchain

In [4]:
%pip install ../../api/python

import IPython; IPython.get_ipython().kernel.do_shutdown(restart=True)

Defaulting to user installation because normal site-packages is not writeable
Processing /home/wallmenis/git/sk-raft-client/api/python
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for skraft: filename=skraft-1.0.0-py3-none-any.whl size=45974 sha256=8908baf90d3c486f055996d490540d4b4d09ffa81630290e35d9da5e2a7c4cab
  Stored in directory: /tmp/pip-ephem-wheel-cache-7zmrlpio/wheels/cc/7e/0f/603cd9eaf8e594f1c766c921daeeef2440d17e0cbb287bfb57
Successfully built skraft
  Attempting uninstall: skraft
    Found existing installation: skraft 1.0.0
    Uninstalling skraft-1.0.0:
      Successfully uninstalled skraft-1.0.0
Note: you may need to restart the kernel to use updated packages.


{'status': 'ok', 'restart': True}

Set up the dependencies

In [1]:
from skraft import BscanPrimitive, Design, SkraftError, SkraftFpga, TargetMemory

Start the server

In [ ]:
%%script bash --bg
sk-raft

Configure the values for the server connection

In [3]:
SK_RAFT_SERVER_IP = "127.0.0.1"
SK_RAFT_SERVER_PORT = 50051

List all the devices and then pick the one you want with its serial number

In [4]:
def main() -> int:
    try:
        # SK-RAFT: open the connection. Then list the connected boards.
        with SkraftFpga.connect(ip=SK_RAFT_SERVER_IP,
                                port=SK_RAFT_SERVER_PORT) as client:
            devices = client.get_devices()
    except SkraftError as e:
        print(f"Cannot list the boards: {e}")
        return 1
    except KeyboardInterrupt:
        print("Interrupted.")
        return 130

    if not devices:
        print("No board is connected to the SK-RAFT server.")
        return 1
    print(f"{'SERIAL NUMBER':16s} PART NUMBER")
    for device in devices:
        print(f"{device.serial_number:16s} {device.part_number}")
    return 0


if __name__ == "__main__":
    main()

SERIAL NUMBER    PART NUMBER
210183BA2524A    XC7A35T


Configure the values for the examples (especially `BOARD` and `SERIAL NUMBER`)

In [8]:
# --------------------------------------------------------------------------- #
# Configuration. Set these values for your setup.
# --------------------------------------------------------------------------- #


# If you do not know the serial number of your target device,
# use the get_devices() call to find it.
SERIAL_NUMBER = "210183BA2524A"

# BOARD is the board of your target device: "basys3" or "zedboard". The
# script reads the design files from
# hw/design_examples/<BOARD>/bram_readback_writeback_top/. Build them with
# create_project.tcl (see README.md). The paths are relative to src/. Run
# the script from that folder.
BOARD = "basys3"
DESIGN_DIR = f"../hw/design_examples/{BOARD}/bram_readback_writeback_top"

# The bitstream programs the device. The LL file gives the location of
# each BRAM content bit.
LL_FILE = f"{DESIGN_DIR}/bram_readback_writeback_top.ll"
design = Design(
    bit=f"{DESIGN_DIR}/bram_readback_writeback_top.bit",
    ll =LL_FILE,
)

# BSCAN registers: (USER number, width in bits)
USER_CLK_EN = BscanPrimitive(1, 1)
USER_ADDRESS = BscanPrimitive(2, 16)
USER_DATA = BscanPrimitive(3, 80)

# The BRAM has 512 words of 64 data bits.
DEPTH = 512

Define some helper functions

In [9]:
# The word that the script writes at each address is the address: word 0
# holds 0, word 1 holds 1, and so on.
def pattern(address: int) -> int:
    return address


# Find the location of a BRAM content bit in the LL file, for example
# "BIT0" (data bit 0 of word 0). Return the frame address (FAR), the word
# in the frame and the bit in the word.
def ll_bit(name: str) -> tuple[int, int, int]:
    with open(LL_FILE, encoding="ascii") as file:
        for line in file:
            fields = line.split()
            if fields[:1] == ["Bit"] and f"Ram=B:{name}" in fields:
                offset = int(fields[3])
                return int(fields[2], 16), offset // 32, offset % 32
    raise ValueError(f"Ram=B:{name} is not in {LL_FILE}")


# The table header and the format of each row.
ROW = "{:<34} {:<18}  {:<6}  {:<7}  {}"


def print_header() -> None:
    print(ROW.format("Stage", "Data", "Parity", "SBITERR",
                     "Data bits that differ from word 0"))


# Read the BRAM outputs through USER3 and print one row: the data, the
# ECC parity, SBITERR and the data bits that differ from the written word
# 0.
def read_word_0(dev, label: str) -> None:
    value = int.from_bytes(dev.read_bscan(USER_DATA), "big")
    data = value & (2**64 - 1)
    parity = (value >> 64) & 0xFF
    sbiterr = (value >> 72) & 1
    diff = [i for i in range(64) if ((data ^ pattern(0)) >> i) & 1]
    bits = ", ".join(str(i) for i in diff) if diff else "none"
    print(ROW.format(label, f"{data:#018x}", f"{parity:#04x}", sbiterr,
                     bits))

Connect to the server

In [10]:
client = SkraftFpga.connect(ip=SK_RAFT_SERVER_IP, port=SK_RAFT_SERVER_PORT)
dev = client.attach(SERIAL_NUMBER)

Program the board and upload the design to SK-RAFT

In [11]:
dev.uploadDesignToServer(design)
dev.programFPGA(design)

Run the example

In [12]:
far, word, bit = ll_bit("BIT0")

print(f"Programmed {SERIAL_NUMBER}")
print(f"Word 0 is written as {pattern(0):#018x}.")
print()
print_header()

# Stage 1: start the clock, fill the BRAM, select word 0
dev.write_bscan(USER_CLK_EN, 1)
for address in range(DEPTH):
    dev.write_bscan(USER_ADDRESS, address)
    dev.write_bscan(USER_DATA, pattern(address))
dev.write_bscan(USER_ADDRESS, 0)
read_word_0(dev, "1  clock on, BRAM filled")

# Stage 2: stop the clock
dev.write_bscan(USER_CLK_EN, 0)
read_word_0(dev, "2  clock stopped")

# Stage 3: inject into word 0, bit 0. SK-RAFT reads the
# content frame, flips the bit and writes the frame back.
dev.inject_at(far, word, bit, TargetMemory.BRAM)
read_word_0(dev, "3  bit 0 injected, clock stopped")

# Stage 4: start the clock. The BRAM reads word 0 again.
dev.write_bscan(USER_CLK_EN, 1)
read_word_0(dev, "4  clock on")

# Program the FPGA again to clear the upset.
dev.programFPGA(design)

Programmed 210183BA2524A
Word 0 is written as 0x0000000000000000.

Stage                              Data                Parity  SBITERR  Data bits that differ from word 0
1  clock on, BRAM filled           0x0000000000000000  0x00    0        none
2  clock stopped                   0x0000000000000000  0x00    0        none
3  bit 0 injected, clock stopped   0x0000000000000003  0x06    0        0, 1
4  clock on                        0x0000000000000001  0x83    0        0


The script prints a table with one row for each stage: the data, the ECC parity, `SBITERR` and the data bits that differ from the written word 0. At the end, it programs the FPGA again to clear the upset.

## Result

Sample output of a run on a Basys 3 (shortened). The frame addresses, the bit counts and the error counts depend on the board, the build and the seed.

```
Programmed 210183BFD452A
Word 0 is written as 0x0000000000000000.

Stage                              Data                Parity  SBITERR  Data bits that differ from word 0
1  clock on, BRAM filled           0x0000000000000000  0x00    0        none
2  clock stopped                   0x0000000000000000  0x00    0        none
3  bit 0 injected, clock stopped   0x0000000000000003  0x06    0        0, 1
4  clock on                        0x0000000000000001  0x83    0        0
```

# Example 4: BRAM injection, configuration bits, content and ECC recovery

The design uses one RAMB36 in the 1K x 36 configuration. Each 36-bit word contains 24 data bits and 12 check bits. Logic in the fabric around the BRAM protects each data byte with its own Hamming SEC code (4 check bits for each byte). This logic is not the built-in ECC of the BRAM.

![Block diagram of brams_top. A BRAM in a pblock has an encoder on the write path. On the read path, a falling-edge hold register on the gated clock feeds a decoder and a multiplexer. USER1 controls the clock gate, USER2 selects the address, USER3 carries the data and USER4 enables the ECC correction.](images/example4_brams_top_abstract.svg)

## Sequence

1. The script fills the memory. Then it reads back each word to make sure that the data is correct.
2. **Part 1.** The script injects essential CRAM upsets (`TargetMemory.ESSENTIAL_CRAM`) into the block type 0 frames of the BRAM column. Then it reads back the memory.
3. The script programs the FPGA again. Thus the upsets of part 1 do not have an effect on part 2.
4. The script fills the memory again.
5. **Part 2.** The script injects content upsets (`TargetMemory.BRAM`) into the block type 1 frames. The upsets add up, and the script does not correct them.
6. The script reads back the full memory two times: first with ECC off, then with ECC on.
7. The script programs the FPGA again, to clear the upsets of part 2.

In the two parts, the script injects one upset each 0.3 s. When part 2 continues for a longer time, more bytes get two or more flips. The SEC code cannot correct these bytes.

## BRAM output latch and the hold register

When SK-RAFT reads or writes BRAM content frames (part 2 injects into them), the output latch of `stock_bram` takes a value from the configuration frame. The content of the BRAM does not change. The BRAM has no output register (`DO_REG = 0`), so without protection the design can take that wrong value instead of the word it read before the freeze.

`hw/create_project.tcl` builds `brams_top` with the BRAM hold register pass ([`auxiliary_components/bram_hold_pass`](../../auxiliary_components/bram_hold_pass/README.md)). The pass adds a falling-edge register behind the BRAM output, which keeps the word read before the freeze. Thus a wrong word in the results comes from the injected content, not from the latch. The build prints the report of the pass.

[`readback_writeback/`](readback_writeback/README.md) shows the effect without the pass, on a BRAM with the built-in ECC: the wrong word while the clock is stopped, and the flipped bit that the ECC does not detect.

## BSCAN register map

| Register | Width | Access | Contents |
| --- | --- | --- | --- |
| USER1 | 1 bit | write | BRAM clock enable (the script stops the clock during each injection) |
| USER2 | 16 bits | write | Address select (only the low 10 bits have an effect) |
| USER3 | 24 bits | read/write (check auxiliary_components/bscan/bscan_user_register_io.vhd to see how reads are performed) | The data at the selected address |
| USER4 | 1 bit | write | ECC correction enable. It applies to the BRAM read path only. The reset value is 0. |

## The calls of the example

| Call | Use in the example |
| --- | --- |
| `SkraftFpga.connect(ip=..., port=...)`, `client.attach(SERIAL_NUMBER)` | Connect to the SK-RAFT server and attach to the device, in `with` statements that release the device and close the connection. |
| `Design(bit, ebd, ll, xdc)`, `dev.uploadDesignToServer(design)`, `dev.programFPGA(design)` | Give the design files, upload them to the server and program the FPGA. |
| `dev.configure_seed(0)` | Set the seed of the random injections. The same seed gives the same upsets in each run. |
| `BscanPrimitive(user, length)`, `dev.write_bscan(primitive, value)`, `dev.read_bscan(primitive)` | Start and stop the clock (USER1), select the address (USER2), write and read the data (USER3), and enable the ECC correction (USER4). |
| `dev.bit_count(TargetMemory.ESSENTIAL_CRAM, pblock=PBLOCK_BRAM)`, `dev.bit_count(TargetMemory.BRAM, pblock=PBLOCK_BRAM)` | Get the number of essential CRAM bits and of content bits of the BRAM pblock. |
| `dev.inject_random(TargetMemory.ESSENTIAL_CRAM, pblock=PBLOCK_BRAM)` | Part 1: inject one upset into a random essential bit of the BRAM pblock. |
| `dev.inject_random(TargetMemory.BRAM, pblock=PBLOCK_BRAM)` | Part 2: inject one upset into a random content bit of the BRAM. |
| `dev.decode_far(far)` | Get the block type of the frame of each upset (0: configuration, 1: BRAM content). |

## Requirements

- Python 3.9 or a later version, with the `skraft` package installed.
- An SK-RAFT server in operation, with the target board connected through USB.
- The bitstream, the EBD file, the LL file and the XDC file that declares `pblock_stock_bram`. The script uploads these files to the server. Use the prebuilt files in `design_examples.zip`, or build them with `hw/create_project.tcl`. Both include the BRAM hold register pass.

## Configuration

| Constant | Default | Description |
| --- | --- | --- |
| `ADDRESS_BITS` | `10` | The address width that has an effect: 1024 words. |

## Result

Sample output of a run on a Basys 3 (shortened). The frame addresses, the bit counts and the error counts depend on the board, the build and the seed.

```
Example 4: BRAM injection, config bits, content, and ECC recovery
Programmed 210183BFD452A.
Exercising the stock BRAM's own configuration before touching its contents.
BRAM Config bit count = 3349
Writing known words in the BRAM...
Wrote 1024 words into pblock_stock_bram.
Checking if all words round-trip...
All 1024 addresses read back correctly.
Press enter to stop injecting
Injecting Essential CRAM bit upsets into pblock_stock_bram's configuration frames (block type 0), none corrected.
     #  FAR       WORD  BIT  BLK_TYPE
     1  00400311    63   29    0
...
Injected 22 Essential CRAM faults.
896 of 1024 addresses read back wrong.
Re-programming to clear the config upsets before the content campaign.
Writing known words in the BRAM...
Wrote 1024 words into pblock_stock_bram.
BRAM Content bit count = 36864
Press enter to stop injecting
Injecting BRAM Content bit upsets into pblock_stock_bram, none corrected.
     #  FAR       WORD  BIT  BLK_TYPE
     1  00c00062    67   10    1
...
Injected 29 BRAM Content faults.
Reading the whole memory with ECC read off.
ECC off: 16 of 1024 addresses wrong.
  ADDR  EXPECTED                  READ                      FAULTY BITS
     1  101001011010010110100100  101001111010010110100100  17
...
16 words wrong, 16 faulty bits in total.
--------------------------------------------------------------------------------
Reading the whole memory again with ECC read on.
ECC on: 0 of 1024 addresses wrong.
ECC on: every word read back correctly. Single flips were corrected; flips in check bits were masked.
Re-programming to clear the device.
Example 4 done.
```

If the first write and read-back pass finds incorrect data, the script stops. It does not inject faults in that case.

After each read-back, the script prints each incorrect word with these data:

- The expected data bits.
- The data bits that it read.
- The positions of the incorrect bits.

The main result is the number of incorrect addresses with ECC off and with ECC on:

- A word that is incorrect only with ECC off had single flips. The SEC code corrected them.
- A word that is incorrect with ECC on had two or more flips in one byte. The SEC code cannot correct these flips, and it can also make an incorrect correction.
- A flip in a check bit does not show with ECC off. With ECC on, the SEC code hides a single flip in a check bit.

The script does not correct the faults. At the end, it programs the FPGA again to clear them.

## Run the example

Import the time library

In [18]:
import time

Change the design and configuration variables

In [19]:
DESIGN_DIR = f"../hw/design_examples/{BOARD}/brams_top"

design = Design(
    bit=f"{DESIGN_DIR}/brams_top.bit",
    ebd=f"{DESIGN_DIR}/brams_top.ebd",
    ll =f"{DESIGN_DIR}/brams_top.ll",
    xdc=f"{DESIGN_DIR}/board.xdc",
)

PBLOCK_BRAM = "pblock_stock_bram"

# BSCAN registers: (USER number, width in bits)
USER_CLK_EN = BscanPrimitive(1, 1)
USER_ADDRESS = BscanPrimitive(2, 16)
USER_DATA = BscanPrimitive(3, 24)
USER_ECC_READ_EN = BscanPrimitive(4, 1)

# The design uses only the 10 low bits of the 16-bit USER2 register.
ADDRESS_BITS = 10
# The BRAM has 1024 words.
DEPTH = 1 << ADDRESS_BITS

DATA_MASK = (1 << USER_DATA.size_in_bits) - 1

Define more helper functions

In [20]:
# The script prints one line for each injected bit. The line shows the
# location of the bit (FAR, word, bit). Then the line shows the block type
# from the decoded FAR (0 = configuration, 1 = BRAM content).
HIT_HEADER = "     #  FAR       WORD  BIT  BLK_TYPE"


def print_hit(dev, n: int, report) -> None:
    frame = dev.decode_far(report.far)
    print(f"{n:>6}  {report.far:08x}  {report.word_index:>4}  "
          f"{report.bit_index:>3}  {frame.blocktype!s:>3}  ")

# Start or stop the BRAM clock through USER1. The script stops the clock
# immediately before each injection below and starts it again immediately
# after. If the clock operates during an injection, the target circuit
# is active when the upset occurs. In a real campaign, the circuit can
# be unstable during the injection routine.
# Use this procedure as a general guideline for all campaigns.
def enable_clock(dev, enabled: bool) -> None:
    dev.write_bscan(USER_CLK_EN, 1 if enabled else 0)

# Select one BRAM address through USER2.
def select(dev, address: int) -> None:
    dev.write_bscan(USER_ADDRESS, address)


# Write the data word into USER3. write_bscan leaves Shift-DR through
# Exit1-DR -> Update-DR. For the design, this TAP path is a write to the
# selected address.
def write_word(dev, value: int) -> None:
    dev.write_bscan(USER_DATA, value)


# Read the data word from USER3. read_bscan leaves Shift-DR through
# Exit1-DR -> Pause-DR and does not go through Update-DR. Thus, the
# design does not write the bits that the read shifts in to the BRAM.
def read_word(dev) -> int:
    data = dev.read_bscan(USER_DATA)
    return int.from_bytes(data, "big") & DATA_MASK


# Start or stop ECC correction through USER4. The design always
# calculates and stores the check bits on each write. USER4 controls
# only the correction on the read path.
def ecc_read_en(dev, enabled: bool) -> None:
    dev.write_bscan(USER_ECC_READ_EN, 1 if enabled else 0)


# Write a known pattern to all words of the BRAM.
def fill_bram(dev) -> None:
    for address in range(DEPTH):
        select(dev, address)
        write_word(dev, (0xA5A5A5 ^ address) & DATA_MASK)
    print(f"Wrote {DEPTH} words into {PBLOCK_BRAM}.")


# Read all words of the BRAM. Return (address, expected, value) for each
# word that is different from the known pattern.
def check_bram(dev) -> list[tuple[int, int, int]]:
    corrupt = []
    # select() and read_word() are separate TAP scans. The second scan
    # cannot start before the first scan is done. The gap between the
    # scans is many TCK cycles. This gap is much longer than the BRAM read
    # latency plus the SEC decode time. Thus, USER3 always captures the
    # word at the new address.
    for address in range(DEPTH):
        select(dev, address)
        value = read_word(dev)
        expected = (0xA5A5A5 ^ address) & DATA_MASK
        if value != expected:
            corrupt.append((address, expected, value))
    return corrupt


# Print each wrong word as the expected bits and the read bits. Also print
# the bit positions (0 = LSB) that changed from the written value.
def print_wrong_words(words: list[tuple[int, int, int]]) -> None:
    if not words:
        return
    width = USER_DATA.size_in_bits
    print(f"{'ADDR':>6}  {'EXPECTED':<{width}}  {'READ':<{width}}  "
          f"FAULTY BITS")
    faulty_bits = 0
    for address, expected, value in words:
        flipped = [bit for bit in range(width)
                   if (expected ^ value) >> bit & 1]
        faulty_bits += len(flipped)
        print(f"{address:>6}  {expected:0{width}b}  {value:0{width}b}  "
              f"{', '.join(str(bit) for bit in flipped)}")
    print(f"{len(words)} words wrong, {faulty_bits} faulty bits in total.")

Program the board and upload the design to the server because it needs it for some operations.

In [21]:
dev.uploadDesignToServer(design)
dev.programFPGA(design)
print(f"programmed {SERIAL_NUMBER} with {design.bit}")

programmed 210183BA2524A with ../hw/design_examples/basys3/brams_top/brams_top.bit


### Run the experiment!
The `N_OF_INJECTIONS` is the number of fault injections the experiment will do.

In [22]:
N_OF_INJECTIONS = 100

# Setup the SEED for deterministic behaviour. 
dev.configure_seed(0)
enable_clock(dev, True)

# Part 1: inject upsets into the configuration frames
# (block type 0) of the BRAM column, not into its content.
# Then program the FPGA again, so no upset from Part 1
# stays in Part 2.
print("Exercising the stock BRAM's own configuration before touching its contents.")

# Note: this count is the size of the set that inject_random
# selects from. The set contains the essential bits in the
# block type 0 frames of the pblock.
bram_config_bits = dev.bit_count(TargetMemory.ESSENTIAL_CRAM, pblock=PBLOCK_BRAM)
print(f"BRAM Config bit count = {bram_config_bits}")  
time.sleep(1)

# USER4 resets to 0 (correction off). The script writes 0 to
# USER4 anyway, so that the code shows the state.
# Read the raw data (no correction) for the round-trip check.
ecc_read_en(dev, False)
# Fill the memory array with known words.
print("Writing known words in the BRAM...")
fill_bram(dev)
# Make sure that each word reads back correctly before the
# example continues.
print("Checking if all words round-trip...")
wrong = len(check_bram(dev))
if wrong:
    print(f"{wrong} of {DEPTH} addresses did not round-trip; not injecting anything.")
    raise SystemExit("Not all words round-trip.")
print(f"All {DEPTH} addresses read back correctly.")

time.sleep(1)
# Inject upsets until you press Enter.
print(f"Injecting Essential CRAM bit upsets into {PBLOCK_BRAM}'s configuration frames (block type 0), none corrected.")
# Each upset is a readback-modify-writeback of a full frame.
# Keep the BRAM idle during the injection (refer to the
# guideline in Part 2).
print(HIT_HEADER)
n = 0
iterator = N_OF_INJECTIONS 
while iterator != 0:
    enable_clock(dev, False)
    report = dev.inject_random(TargetMemory.ESSENTIAL_CRAM, pblock=PBLOCK_BRAM)
    enable_clock(dev, True)
    print_hit(dev, n + 1, report)
    time.sleep(0.3)
    iterator -= 1
    n += 1

print(f"Injected {n} Essential CRAM faults.")
# Scan the memory to count the wrong words.
errors = len(check_bram(dev))
# A wrong read-back shows that an upset hit the BRAM or its
# access path. The read-back cannot show which of the two,
# because the frames also hold interconnect.
if errors == DEPTH:
    print(f"All {DEPTH} addresses read back wrong: the BRAM or its access path no longer works.")
elif errors:
    print(f"{errors} of {DEPTH} addresses read back wrong.")
else:
    print("All words still read back correctly: the upsets hit bits this read/write path doesn't depend on.")

Exercising the stock BRAM's own configuration before touching its contents.
BRAM Config bit count = 3349
Writing known words in the BRAM...
Wrote 1024 words into pblock_stock_bram.
Checking if all words round-trip...
All 1024 addresses read back correctly.
Injecting Essential CRAM bit upsets into pblock_stock_bram's configuration frames (block type 0), none corrected.
     #  FAR       WORD  BIT  BLK_TYPE
     1  00400311    63   29    0  
     2  0040030f    65   29    0  
     3  00400317    63   22    0  
     4  0040031b    67   31    0  
     5  00400312    64   26    0  
     6  0040031b    61   24    0  
     7  00400316    66   20    0  
     8  00400313    66   10    0  
     9  00400310    64    2    0  
    10  0040031a    69    7    0  
    11  00400309    70   21    0  
    12  00400313    65   16    0  
    13  00400317    67    9    0  
    14  0040030a    63   13    0  
    15  0040030f    67   27    0  
    16  0040030a    68   21    0  
    17  00400314    66   21    

In [23]:
print("Re-programming to clear the config upsets before the content campaign.")
dev.programFPGA(design)

Re-programming to clear the config upsets before the content campaign.


In [24]:
N_OF_INJECTIONS = 100

# Setup the SEED for deterministic behaviour. 
dev.configure_seed(0)

enable_clock(dev, True)

# When the script programs the FPGA again, USER4 resets to 0.
# Thus, the correction is off again. The script does not do the
# round-trip check again. The check in Part 1 showed that the
# BRAM write/read path is stable.

# BRAM content: fill the memory with known words.
print("Writing known words in the BRAM...")
fill_bram(dev)
# Note: this count is the size of the set that inject_random
# selects from. The set contains all data bits and check
# bits of the BRAM (1K x 36 words).
# The upsets below accumulate. The script does not correct
# them.
bram_content_bits = dev.bit_count(TargetMemory.BRAM,
                                                  pblock=PBLOCK_BRAM)
print(f"BRAM Content bit count = {bram_content_bits}")
                
time.sleep(1)
print(f"Injecting BRAM Content bit upsets into {PBLOCK_BRAM}, none corrected.")
print(HIT_HEADER)
n = 0
iterator = N_OF_INJECTIONS
while iterator != 0:
    # Injection guideline: each content upset is a
    # readback-modify-writeback of a full frame of block
    # type 1. That frame holds bits of more than one BRAM in
    # the column. The server does not protect against user
    # traffic. Thus, keep all BRAMs in the target frames
    # idle (no reads or writes) during the injection.
    enable_clock(dev, False)
    report = dev.inject_random(TargetMemory.BRAM, 
                                               pblock=PBLOCK_BRAM)
    enable_clock(dev, True)
    print_hit(dev, n + 1, report)
    time.sleep(0.3)
    n += 1
    iterator -= 1

print(f"Injected {n} BRAM Content faults.")  
print("Reading the whole memory with ECC read off.")
# Scan the memory to find the wrong words.
err_words = check_bram(dev)
print(f"ECC off: {len(err_words)} of {DEPTH} addresses wrong.")
print_wrong_words(err_words)
print("-" * 80)
# Start the ECC correction on the read path.
ecc_read_en(dev, True)
print("Reading the whole memory again with ECC read on.")
# Scan the memory to find the wrong words.
corrupt_w_ecc = check_bram(dev)
print(f"ECC on: {len(corrupt_w_ecc)} of {DEPTH} addresses "
                      f"wrong.")
print_wrong_words(corrupt_w_ecc)

# The read path gives only the corrected data, with no error
# flag. With ECC off, flipped check bits do not show. With
# ECC on, SEC masks a single flipped check bit. SEC cannot
# detect two or more flipped bits in one byte. Then SEC can
# change a bit that is correct. This adds an error and
# does not remove one.
wrong_ecc_off = {address for address, _, _ in err_words}
wrong_ecc_on = {address for address, _, _ in corrupt_w_ecc}
if not wrong_ecc_on:
    print("ECC on: every word read back correctly. Single "
                          "flips were corrected; flips in check bits were "
                          "masked.")
else:
    print(f"ECC on: {len(wrong_ecc_on)} address(es) still wrong - "
                          f"at least one byte had two or more flips, which "
                          f"SEC can't correct.")
new = wrong_ecc_on - wrong_ecc_off
if new:
    print(f"{len(new)} of them read correctly with ECC "
                              f"off: the flips were in check bits and the "
                              f"decoder mis-corrected a data bit.")

Writing known words in the BRAM...
Wrote 1024 words into pblock_stock_bram.
BRAM Content bit count = 36864
Injecting BRAM Content bit upsets into pblock_stock_bram, none corrected.
     #  FAR       WORD  BIT  BLK_TYPE
     1  00c00055    63   26    1  
     2  00c00020    61   18    1  
     3  00c0001c    64   25    1  
     4  00c00061    61   10    1  
     5  00c00040    69   27    1  
     6  00c0003a    67    0    1  
     7  00c0002e    63   15    1  
     8  00c0006e    68    5    1  
     9  00c00076    63    8    1  
    10  00c0000d    62   16    1  
    11  00c0002a    67    8    1  
    12  00c00045    66   27    1  
    13  00c00025    61   11    1  
    14  00c00010    66   28    1  
    15  00c0003f    61   24    1  
    16  00c00044    63   14    1  
    17  00c00070    67   31    1  
    18  00c0002a    61    2    1  
    19  00c00030    69    2    1  
    20  00c00007    64   30    1  
    21  00c00026    68   11    1  
    22  00c0003e    63   20    1  
    23  00c

In [ ]:
print("Re-programming to clear the device.")
dev.programFPGA(design)